# FinOps anomaly detector

This notebook uses an unsupervised + statistical engine to discover anomalies from the data itself:
- robust statistics with rolling median and MAD
- unsupervised outlier detection with Isolation Forest when available
- rule-based filters for sustained spikes, low-usage high-cost resources, and missing tags

Important note:
- The file anomaly_labels_full.csv is used only for evaluation and reporting.
- It is not used for training or supervised tuning.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

try:
    from sklearn.ensemble import IsolationForest
    from sklearn.preprocessing import StandardScaler
    HAS_SKLEARN = True
except Exception:
    HAS_SKLEARN = False

ROOT = Path.cwd()
if not (ROOT / "data" / "cost_explorer_daily.csv").exists():
    ROOT = Path("h:/Capstone_Phase2_AI")

DAILY_PATH = ROOT / "data" / "cost_explorer_daily.csv"
LINE_ITEMS_PATH = ROOT / "data" / "cur_line_items.csv"
LABELS_PATH = ROOT / "anomaly_labels_full.csv"
OUTPUT_PATH = ROOT / "detected_alerts.csv"
SUMMARY_PATH = ROOT / "summary_report.json"
EVAL_PATH = ROOT / "evaluation_report.json"

print("Root:", ROOT)
print("Daily file exists:", DAILY_PATH.exists())
print("Line-items file exists:", LINE_ITEMS_PATH.exists())
print("Labels file exists:", LABELS_PATH.exists())
print("Scikit-learn available:", HAS_SKLEARN)

In [ ]:
daily_df = pd.read_csv(DAILY_PATH)
daily_df["date"] = pd.to_datetime(daily_df["date"]).dt.normalize()
daily_df = daily_df.sort_values(["linked_account_id", "service", "date"]).reset_index(drop=True)

line_df = pd.read_csv(LINE_ITEMS_PATH)
line_df["date"] = pd.to_datetime(line_df["line_item_usage_start_date"]).dt.normalize()
line_df["line_item_unblended_cost"] = pd.to_numeric(line_df["line_item_unblended_cost"], errors="coerce")
line_df["line_item_usage_amount"] = pd.to_numeric(line_df["line_item_usage_amount"], errors="coerce")
line_df["resource_id"] = line_df["line_item_resource_id"].fillna("unknown")
line_df["team_tag"] = line_df["resource_tags_user_team"].fillna("")
line_df["owner_tag"] = line_df["resource_tags_user_owner"].fillna("")
line_df["service"] = line_df["product_product_name"].fillna("unknown")

labels_df = pd.read_csv(LABELS_PATH)
labels_df["start_date"] = pd.to_datetime(labels_df["start_date"]).dt.normalize()
labels_df["end_date"] = pd.to_datetime(labels_df["end_date"]).dt.normalize()

print("Daily rows:", len(daily_df))
print("Line-item rows:", len(line_df))
print("Reference labels:", len(labels_df))

In [ ]:
BENIGN_RESOURCE_PATTERNS = ("loadtest", "flashsale", "autoscale", "migration", "sandbox", "staging", "test")


def is_benign_resource(resource_id: object, service: object) -> bool:
    resource_text = f"{resource_id} {service}".lower()
    return any(pattern in resource_text for pattern in BENIGN_RESOURCE_PATTERNS)


def add_robust_features(group):
    group = group.sort_values("date").copy()
    prev_cost = group["cost"].shift(1)
    prev_usage = group["usage"].shift(1)
    baseline_cost = prev_cost.rolling(14, min_periods=7).median()
    mad_cost = prev_cost.rolling(14, min_periods=7).apply(lambda x: np.median(np.abs(x - np.median(x))), raw=True)
    std_cost = prev_cost.rolling(14, min_periods=7).std()
    group["baseline_cost"] = baseline_cost
    group["mad_cost"] = mad_cost
    group["std_cost"] = std_cost
    group["cost_ratio_to_baseline"] = group["cost"] / baseline_cost.replace(0, np.nan)
    group["mad_score"] = 0.6745 * (group["cost"] - baseline_cost) / mad_cost.replace(0, np.nan)
    group["z_score"] = (group["cost"] - baseline_cost) / std_cost.replace(0, np.nan)
    group["daily_change_pct"] = group["cost"].pct_change()
    group["usage_change_pct"] = prev_usage.replace(0, np.nan).pipe(lambda s: (group["usage"] - s) / s)
    group["daily_change_pct"] = group["daily_change_pct"].replace([np.inf, -np.inf], np.nan).fillna(0)
    group["usage_change_pct"] = group["usage_change_pct"].replace([np.inf, -np.inf], np.nan).fillna(0)
    group["cost_ratio_to_baseline"] = group["cost_ratio_to_baseline"].replace([np.inf, -np.inf], np.nan)
    group["mad_score"] = group["mad_score"].replace([np.inf, -np.inf], np.nan).fillna(0)
    group["z_score"] = group["z_score"].replace([np.inf, -np.inf], np.nan).fillna(0)
    group["low_usage_high_cost"] = (group["usage"] <= 1) & (group["cost"] >= 80)
    group["tag_missing_flag"] = group["tag_missing"] & (group["cost"] >= 100)
    return group

resource_daily = (
    line_df.groupby(["resource_id", "line_item_usage_account_id", "line_item_usage_account_name", "service", "date", "team_tag", "owner_tag"], as_index=False)
    .agg(cost=("line_item_unblended_cost", "sum"), usage=("line_item_usage_amount", "sum"))
)
resource_daily = resource_daily.sort_values(["resource_id", "date"]).reset_index(drop=True)
resource_daily["tag_missing"] = resource_daily["team_tag"].fillna("").str.strip().eq("") | resource_daily["owner_tag"].fillna("").str.strip().eq("")

feature_frames = []
for _, group in resource_daily.groupby("resource_id", dropna=False):
    feature_frames.append(add_robust_features(group))

resource_features = pd.concat(feature_frames, ignore_index=True)
resource_features["mad_score"] = resource_features["mad_score"].fillna(0)
resource_features["z_score"] = resource_features["z_score"].fillna(0)
resource_features["daily_change_pct"] = resource_features["daily_change_pct"].fillna(0)
resource_features["usage_change_pct"] = resource_features["usage_change_pct"].fillna(0)
resource_features["is_benign"] = resource_features.apply(lambda row: is_benign_resource(row["resource_id"], row["service"]), axis=1)

feature_cols = ["cost", "cost_ratio_to_baseline", "mad_score", "z_score", "daily_change_pct", "usage_change_pct", "low_usage_high_cost", "tag_missing_flag"]
model_input = resource_features[feature_cols].fillna(0)

if HAS_SKLEARN:
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(model_input)
    iso = IsolationForest(contamination=0.02, random_state=42, n_estimators=300)
    preds = iso.fit_predict(X_scaled)
    resource_features["isoforest_anomaly"] = (preds == -1)
    resource_features["isoforest_score"] = -iso.score_samples(X_scaled)
else:
    resource_features["isoforest_anomaly"] = False
    resource_features["isoforest_score"] = 0.0

resource_features["strong_spike"] = (
    (resource_features["cost_ratio_to_baseline"] >= 1.4)
    & (resource_features["z_score"].abs() >= 1.5)
    & (resource_features["cost"] >= 50)
)
resource_features["strong_drift"] = (
    (resource_features["cost_ratio_to_baseline"] >= 1.25)
    & (resource_features["daily_change_pct"].abs() >= 0.10)
    & (resource_features["cost"] >= 60)
)
resource_features["strong_idle"] = resource_features["low_usage_high_cost"] & (resource_features["cost"] >= 60)
resource_features["strong_untagged"] = (
    resource_features["tag_missing_flag"]
    & (resource_features["cost"] >= 80)
    & ((resource_features["cost_ratio_to_baseline"] >= 1.2) | (resource_features["z_score"].abs() >= 1.5))
)
resource_features["strong_signal"] = (
    resource_features["strong_spike"]
    | resource_features["strong_drift"]
    | resource_features["strong_idle"]
    | resource_features["strong_untagged"]
)

resource_features["score"] = (
    np.clip((resource_features["cost_ratio_to_baseline"] - 1.0) / 2.0, 0, 1) * 0.30
    + np.clip(resource_features["mad_score"].abs() / 4.0, 0, 1) * 0.15
    + np.clip(resource_features["z_score"].abs() / 4.0, 0, 1) * 0.20
    + np.clip(resource_features["daily_change_pct"].abs() / 1.5, 0, 1) * 0.15
    + (resource_features["low_usage_high_cost"] * 0.10)
    + (resource_features["tag_missing_flag"] * 0.05)
    + (resource_features["isoforest_anomaly"] * 0.05)
)
resource_features["score"] = resource_features["score"].fillna(0)
resource_features["candidate"] = (
    (~resource_features["is_benign"])
    & (resource_features["score"] >= 0.40)
    & (resource_features["strong_signal"] | (resource_features["isoforest_anomaly"] & (resource_features["score"] >= 0.60)))
)

alert_rows = []
for resource_id, group in resource_features.groupby("resource_id", dropna=False):
    group = group.sort_values("date").copy()
    group["candidate_start"] = group["candidate"] & (~group["candidate"].shift(fill_value=False))
    group["event_id"] = group["candidate_start"].cumsum()
    for _, event in group[group["candidate"]].groupby("event_id", dropna=False):
        event = event.sort_values("date")
        peak = event.sort_values(["score", "cost"], ascending=False).iloc[0]
        if peak["score"] < 0.40:
            continue
        event_len = len(event)
        peak_cost = float(event["cost"].max())
        peak_ratio = float(event["cost_ratio_to_baseline"].max())
        peak_z = float(event["z_score"].abs().max())

        if peak["is_benign"]:
            continue
        if peak["strong_idle"] and peak_cost >= 60 and event_len >= 2:
            anomaly_type = "idle_resource"
        elif peak["strong_spike"] and peak_ratio >= 1.4 and peak_z >= 1.5 and event_len >= 2:
            anomaly_type = "sudden_spike"
        elif peak["strong_drift"] and event_len >= 2:
            anomaly_type = "gradual_drift"
        elif peak["strong_untagged"] and peak_cost >= 80 and event_len >= 2 and (peak_ratio >= 1.2 or peak_z >= 1.5):
            anomaly_type = "untagged_spend"
        else:
            continue

        severity = "high" if peak["score"] >= 0.70 or peak_cost >= 200 else "medium"
        fp_risk = "low" if anomaly_type != "untagged_spend" else "medium"
        reason = (
            "Robust-statistical deviation from the rolling baseline."
            if anomaly_type in {"sudden_spike", "gradual_drift"}
            else "Low-usage resource still incurs cost."
            if anomaly_type == "idle_resource"
            else "High-cost resource is missing ownership or team tags."
        )
        alert_rows.append({
            "detected_at": peak["date"].strftime("%Y-%m-%d"),
            "account": str(peak["line_item_usage_account_id"]),
            "service": peak["service"],
            "resource_id": str(peak["resource_id"]),
            "anomaly_type": anomaly_type,
            "severity": severity,
            "score": round(float(peak["score"]), 3),
            "fp_risk": fp_risk,
            "reason": reason,
            "source": "unsupervised-statistical"
        })

alerts_df = pd.DataFrame(alert_rows)
if not alerts_df.empty:
    alerts_df = alerts_df.sort_values(["account", "service", "resource_id", "detected_at", "score"], ascending=[True, True, True, True, False]).reset_index(drop=True)
    alerts_df = alerts_df.drop_duplicates(subset=["account", "service", "resource_id", "detected_at", "anomaly_type"], keep="first")

alerts_df.head()

In [ ]:
if alerts_df.empty:
    alerts_df = pd.DataFrame(columns=["detected_at", "account", "service", "resource_id", "anomaly_type", "severity", "score", "fp_risk", "reason", "source"])

alerts_df.to_csv(OUTPUT_PATH, index=False)

reference_matches = []
for _, pred in alerts_df.iterrows():
    account = str(pred["account"])
    resource_id = str(pred["resource_id"])
    detected_at = pd.to_datetime(pred["detected_at"]).normalize()
    match = labels_df[
        (labels_df["linked_account_id"].astype(str) == account)
        & (labels_df["resource_id"].astype(str) == resource_id)
        & (labels_df["start_date"] <= detected_at)
        & (labels_df["end_date"] >= detected_at)
    ]
    if not match.empty:
        reference_matches.append(True)

precision = len(reference_matches) / len(alerts_df) if len(alerts_df) else 0.0
recall = len(reference_matches) / len(labels_df) if len(labels_df) else 0.0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

summary = {
    "alerts_detected": int(len(alerts_df)),
    "reference_labels": int(len(labels_df)),
    "matched_reference_labels": int(len(reference_matches)),
    "precision": round(precision, 3),
    "recall": round(recall, 3),
    "f1": round(f1, 3),
    "evaluation_note": "Reference labels are used only for evaluation and reporting."
}

with SUMMARY_PATH.open("w", encoding="utf-8") as fh:
    json.dump(summary, fh, ensure_ascii=False, indent=2)

with EVAL_PATH.open("w", encoding="utf-8") as fh:
    json.dump(summary, fh, ensure_ascii=False, indent=2)

summary

## Next steps

- Review the generated alerts in the output CSV.
- Tune the thresholds for MAD, z-score, and contamination if you want fewer or more alerts.
- Keep the reference labels strictly for evaluation and reporting.